# 130 · Fix the environment — solution

**Level:** basic · **Topics:** environment, installation, applications-library, HDAWG, flux · **Type:** fix the bugs

Solution of [exercise.ipynb](exercise.ipynb). The code is the same as in the exercise: only the environment changes.

In [ ]:
import numpy as np

from laboneq.dsl.quantum import QPU
from laboneq.simple import *
from laboneq_applications.experiments import amplitude_rabi
from laboneq_applications.qpu_types.tunable_transmon import (
    TunableTransmonOperations,
    TunableTransmonQubit,
)

## Device setup

Drive and readout on the SHFQC+, the flux line on the HDAWG, synchronised by a PQSC.

📖 [Device setup](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/index.html)

In [ ]:
setup = DeviceSetup("exercise_setup")
setup.add_dataserver(host="localhost", port="8004")
setup.add_instruments(
    SHFQC(uid="shfqc", address="DEV12001", device_options="SHFQC/PLUS/QC2CH"),
    HDAWG(uid="hdawg", address="DEV8001", device_options="HDAWG8/MF/ME/SKW/PC"),
    PQSC(uid="pqsc", address="DEV10001", reference_clock_source="internal"),
)
setup.add_connections(
    "shfqc",
    create_connection(to_signal="q0/drive", ports="SGCHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/measure", ports="QACHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/acquire", ports="QACHANNELS/0/INPUT"),
)
setup.add_connections(
    "hdawg",
    create_connection(to_signal="q0/flux", ports="SIGOUTS/0"),
)
setup.add_connections(
    "pqsc",
    create_connection(to_instrument="shfqc", ports="ZSYNCS/0"),
    create_connection(to_instrument="hdawg", ports="ZSYNCS/1"),
)

## Qubit and calibration

The calibration is generated from the qubit parameters.

📖 [Calibrating signals](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/02_logical_signals/concepts/00_calibrating_signals.html) ·
[Quantum operations](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/04_quantum_processing_unit/tutorials/00_quantum_operations.html)

In [ ]:
[q0] = TunableTransmonQubit.from_device_setup(setup)
q0.update(
    drive_lo_frequency=4.0e9,
    resonance_frequency_ge=4.1e9,
    readout_lo_frequency=7.0e9,
    readout_resonator_frequency=7.08e9,
)

qpu = QPU(quantum_elements=[q0], quantum_operations=TunableTransmonOperations())
setup.set_calibration(q0.calibration())

## Session

📖 [Session](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/01_session/concepts/index.html)

In [ ]:
session = Session(setup)
session.connect(do_emulation=True)

## Amplitude Rabi

A standard experiment of the Applications Library.

📖 [Amplitude Rabi workflow](https://docs.zhinst.com/labone_q_user_manual/applications_library/how-to-guides/sources/01_superconducting_qubits/01_workflows/04_amplitude_rabi.html)

In [ ]:
options = amplitude_rabi.experiment_workflow.options()
options.count(16)
options.do_analysis(False)

rabi = amplitude_rabi.experiment_workflow(
    session, qpu, "q0", amplitudes=np.linspace(0, 1, 11), options=options
)
result = rabi.run()

In [ ]:
# check
print(result.output["q0/result"].data.shape)

**Expected result:** `(11,)`

## Diagnosis

The first thing to check, and to send along with any support request: the versions.

In [ ]:
from importlib.metadata import version

print("laboneq:", version("laboneq"))
print("laboneq-applications:", version("laboneq-applications"))

In the broken environment this prints `laboneq: 26.10.1` and `laboneq-applications: 26.7.0b1`.
The two packages are released together and must have the **same version**.

The flux line shows the consequence. In the broken environment it has a local oscillator, here it has none:

In [ ]:
print(q0.calibration()["q0/flux"])

## The fix

In `environment/pyproject.toml`, use the same version for both packages and remove the override:

```toml
dependencies = [
    "laboneq==26.10.1",
    "laboneq_applications==26.10.1",
    "matplotlib",
    "jupyterlab",
    "ipykernel",
]
```

and delete the whole `[tool.uv]` section. Then `uv sync`, restart the kernel, run again.

In an environment managed with pip: `pip install -U "laboneq==26.10.1" "laboneq-applications==26.10.1"`.

## Answers

1. `laboneq` 26.10.1 with `laboneq-applications` 26.7.0b1.
2. It does not, by itself: `laboneq-applications` 26.7.0b1 requires `laboneq` 26.7.0b1, and `uv sync` refuses the combination. The `override-dependencies` entry in `[tool.uv]` forces it. With pip the same happens when only one of the two packages is upgraded: pip prints a dependency warning and installs it anyway.
3. The old Applications Library always puts a local oscillator in the calibration of the flux line, and the LabOne Q of that time ignored it on an HDAWG. LabOne Q 26.10 rejects it. The matching Applications Library only sets it when the flux line is on a signal generator channel (qubit parameter `flux_on_sg_channel`).
4. No. It hides this one symptom, and the two packages still do not belong together: the next difference will show up somewhere else. Compatibility is only tested between equal versions.

**Takeaway:** when an error makes no sense, print the versions first. `laboneq` and `laboneq-applications` must have the same version.